In [1]:
import xarray as xr
import pandas as pd
import geopandas as gpd

In [2]:
path = '/data/keeling/a/rytam2/a/iema_output/arcgis_toprocess/feb14_2026/'
ds = xr.open_mfdataset(path+'daily_wbgt_by_county_2016-2025.nc').drop_vars('county')

1. convert to -180:180
2. flatten as dataframe

In [22]:
ds

<xarray.Dataset> Size: 80MB
Dimensions:     (time: 3653, lat: 29, lon: 27)
Coordinates:
  * lat         (lat) float32 116B 43.25 43.0 42.75 42.5 ... 36.75 36.5 36.25
  * time        (time) datetime64[ns] 29kB 2016-01-01 2016-01-02 ... 2025-12-31
  * lon         (lon) float32 108B -92.75 -92.5 -92.25 ... -86.75 -86.5 -86.25
Data variables:
    mean        (time, lat, lon) float32 11MB dask.array<chunksize=(3653, 29, 27), meta=np.ndarray>
    max         (time, lat, lon) float32 11MB dask.array<chunksize=(3653, 29, 27), meta=np.ndarray>
    min         (time, lat, lon) float32 11MB dask.array<chunksize=(3653, 29, 27), meta=np.ndarray>
    hr_over_80  (time, lat, lon) int64 23MB dask.array<chunksize=(3653, 29, 27), meta=np.ndarray>
    hr_over_85  (time, lat, lon) int64 23MB dask.array<chunksize=(3653, 29, 27), meta=np.ndarray>

In [3]:
if ds["lon"].values.max() > 180:
    ds = ds.assign_coords(lon=(ds["lon"] % 360 + 540) % 360 - 180)
    ds = ds.sortby("lon")

df = ds.to_dataframe().reset_index()
# df

In [4]:
gdf_points = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["lon"], df["lat"]),
    crs="EPSG:4326"
)

In [34]:
gdf_points

,time,lat,lon,mean,max,min,hr_over_80,hr_over_85,geometry
0,2016-01-01,43.25,-92.75,17.170675,23.309761,9.582619,0,0,POINT (-92.75 43.25)
1,2016-01-01,43.25,-92.50,17.027386,23.308552,9.322681,0,0,POINT (-92.5 43.25)
2,2016-01-01,43.25,-92.25,16.763329,22.971546,9.078291,0,0,POINT (-92.25 43.25)
3,2016-01-01,43.25,-92.00,16.961084,23.307232,9.482477,0,0,POINT (-92 43.25)
4,2016-01-01,43.25,-91.75,17.349447,23.864624,10.145393,0,0,POINT (-91.75 43.25)
...,...,...,...,...,...,...,...,...,...
2860294,2025-12-31,36.25,-87.25,29.790380,41.175285,23.268341,0,0,POINT (-87.25 36.25)
2860295,2025-12-31,36.25,-87.00,29.854795,41.307781,23.361506,0,0,POINT (-87 36.25)
2860296,2025-12-31,36.25,-86.75,30.146357,41.706257,23.629242,0,0,POINT (-86.75 36.25)
2860297,2025-12-31,36.25,-86.50,30.266340,41.680878,23.908075,0,0,POINT (-86.5 36.25)


In [11]:
gdf_counties = gpd.read_file("/data/keeling/a/rytam2/climstat/local/county_zc_shpfile/county/tl_2025_us_county.shp")

gdf_counties_il = gdf_counties[gdf_counties['STATEFP'] == '17'].reset_index(drop=True)

print(gdf_counties_il.crs)
print(gdf_counties_il.columns)  # <-- paste this output so we can confirm FIPS/county col names

if gdf_counties_il.crs.to_epsg() != 4326:
    gdf_counties_il = gdf_counties_il.to_crs("EPSG:4326")


EPSG:4269
Index(['STATEFP', 'COUNTYFP', 'COUNTYNS', 'GEOID', 'GEOIDFQ', 'NAME',
       'NAMELSAD', 'LSAD', 'CLASSFP', 'MTFCC', 'CSAFP', 'CBSAFP', 'METDIVFP',
       'FUNCSTAT', 'ALAND', 'AWATER', 'INTPTLAT', 'INTPTLON', 'geometry'],
      dtype='object')


In [12]:
gdf_counties_il#['COUNTYFP'].unique()

,STATEFP,COUNTYFP,COUNTYNS,GEOID,GEOIDFQ,NAME,NAMELSAD,LSAD,CLASSFP,MTFCC,CSAFP,CBSAFP,METDIVFP,FUNCSTAT,ALAND,AWATER,INTPTLAT,INTPTLON,geometry
0,17,015,00424209,17015,0500000US17015,Carroll,Carroll County,06,H1,G4020,None,None,None,A,1153597409,55881823,+42.0708996,-089.9241898,"POLYGON ((-89.97844 41.93119, -89.97845 41.931..."
1,17,043,00422191,17043,0500000US17043,DuPage,DuPage County,06,H1,G4020,176,16980,16984,A,849236415,22006224,+41.8520577,-088.0860389,"POLYGON ((-87.92019 41.95326, -87.92019 41.953..."
2,17,133,01784865,17133,0500000US17133,Monroe,Monroe County,06,H1,G4020,476,41180,None,A,997924032,33756312,+38.2779831,-090.1790777,"POLYGON ((-89.98459 38.30841, -89.9843 38.3084..."
3,17,157,01784967,17157,0500000US17157,Randolph,Randolph County,06,H1,G4020,None,None,None,A,1490194102,56150220,+38.0565149,-089.8212096,"POLYGON ((-89.59507 37.95534, -89.59791 37.950..."
4,17,149,01784941,17149,0500000US17149,Pike,Pike County,06,H1,G4020,None,None,None,A,2153190712,45251538,+39.6251059,-090.8890344,"POLYGON ((-90.91667 39.84492, -90.91661 39.844..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
97,17,007,00424205,17007,0500000US17007,Boone,Boone County,06,H1,G4020,466,40420,None,A,727137091,3338156,+42.3189831,-088.8242951,"POLYGON ((-88.70547 42.24077, -88.70543 42.233..."
98,17,137,00424270,17137,0500000US17137,Morgan,Morgan County,06,H1,G4020,522,27300,None,A,1473595806,9053547,+39.7176572,-090.2049929,"POLYGON ((-90.08049 39.52104, -90.0876 39.5209..."
99,17,161,00424282,17161,0500000US17161,Rock Island,Rock Island County,06,H1,G4020,209,19340,None,A,1107212317,61631338,+41.4684205,-090.5721252,"POLYGON ((-90.33554 41.67248, -90.3355 41.6729..."
100,17,111,01784815,17111,0500000US17111,McHenry,McHenry County,06,H1,G4020,176,16980,16984,A,1562806157,19855932,+42.3242982,-088.4522450,"POLYGON ((-88.35441 42.154, -88.35929 42.154, ..."


In [13]:
gdf_joined = gpd.sjoin(
    gdf_points,
    gdf_counties_il[["GEOID", "NAMELSAD", "geometry"]],
    how="left", 
    predicate="intersects" #instead of 'within' - returns null for points on county/zc borders
)

gdf_joined = gdf_joined.rename(columns={"NAME": "county"})


In [14]:
gdf_joined

,time,lat,lon,mean,max,min,hr_over_80,hr_over_85,geometry,index_right,GEOID,NAMELSAD
0,2016-01-01,43.25,-92.75,17.170675,23.309761,9.582619,0,0,POINT (-92.75 43.25),NaN,NaN,NaN
1,2016-01-01,43.25,-92.50,17.027386,23.308552,9.322681,0,0,POINT (-92.5 43.25),NaN,NaN,NaN
2,2016-01-01,43.25,-92.25,16.763329,22.971546,9.078291,0,0,POINT (-92.25 43.25),NaN,NaN,NaN
3,2016-01-01,43.25,-92.00,16.961084,23.307232,9.482477,0,0,POINT (-92 43.25),NaN,NaN,NaN
4,2016-01-01,43.25,-91.75,17.349447,23.864624,10.145393,0,0,POINT (-91.75 43.25),NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
2860294,2025-12-31,36.25,-87.25,29.790380,41.175285,23.268341,0,0,POINT (-87.25 36.25),NaN,NaN,NaN
2860295,2025-12-31,36.25,-87.00,29.854795,41.307781,23.361506,0,0,POINT (-87 36.25),NaN,NaN,NaN
2860296,2025-12-31,36.25,-86.75,30.146357,41.706257,23.629242,0,0,POINT (-86.75 36.25),NaN,NaN,NaN
2860297,2025-12-31,36.25,-86.50,30.266340,41.680878,23.908075,0,0,POINT (-86.5 36.25),NaN,NaN,NaN


In [15]:
## Check 
# Check for unmatched ERA5 points (fell outside all counties)
unmatched_points = gdf_joined["GEOID"].isna().sum()
print(f"Unmatched ERA5 points: {unmatched_points}")

# Check for counties with no ERA5 point inside them
matched_counties = gdf_joined["GEOID"].nunique()
total_counties = gdf_counties["GEOID"].nunique()
print(f"Counties with no ERA5 point: {total_counties - matched_counties}")

Unmatched ERA5 points: 1961661
Counties with no ERA5 point: 3134


In [17]:
invalid = gdf_counties_il[~gdf_counties_il.is_valid]
print(f"Invalid geometries: {len(invalid)}")


Invalid geometries: 0


In [20]:
from shapely.geometry import Point

test_point_gdf = gpd.GeoDataFrame(
    {"latitude": [40.0], "longitude": [-89.0]},
    geometry=[Point(-89.0, 40.0)],
    crs="EPSG:4326"
)

result = gpd.sjoin(
    test_point_gdf,
    gdf_counties_il[["GEOID", "NAMELSAD", "geometry"]],
    how="left",
    predicate="intersects"
)
print(result[["latitude", "longitude", "GEOID", "NAMELSAD"]])

   latitude  longitude  GEOID      NAMELSAD
0      40.0      -89.0  17115  Macon County


In [21]:
print(type(gdf_points.geometry.iloc[0]))
print(gdf_points.geometry.iloc[0])
print(gdf_points.geometry.is_valid.all())
print(gdf_points.geometry.is_empty.any())

<class 'shapely.geometry.point.Point'>
POINT (-92.75 43.25)
True
False


In [24]:
print(f"Unique lat/lon combinations: {df[['lat','lon']].drop_duplicates().shape[0]}")
print(f"Total rows in gdf_points: {len(gdf_points)}")
print(f"Number of timesteps: {df['time'].nunique()}")


Unique lat/lon combinations: 783
Total rows in gdf_points: 2860299
Number of timesteps: 3653


In [25]:
2860299/3653

783.0